# Data Preparation for LLM Fine-Tuning

This notebook demonstrates how to prepare data for fine-tuning language models. Proper data preparation is crucial for successful fine-tuning and can significantly impact the performance of your model.

## Why Data Preparation Matters

- **Quality**: High-quality data leads to high-quality models
- **Format**: Different fine-tuning approaches require specific data formats
- **Size**: The amount of data needed depends on the task and model size
- **Balance**: Balanced datasets prevent biased models
- **Preprocessing**: Proper tokenization and formatting improves training efficiency

## Setup Environment

First, let's install the required packages:

In [ ]:
!pip install -q transformers datasets pandas nltk scikit-learn

## Import Libraries

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import wordnet
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# Download WordNet for thesaurus data
nltk.download('wordnet')
nltk.download('omw-1.4')

## 1. Data Collection

Let's start by collecting thesaurus data from WordNet:

In [ ]:
def get_synonyms(word):
    """Get synonyms for a word from WordNet."""
    synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name() != word and lemma.name() not in synonyms:
                synonyms.append(lemma.name().replace('_', ' '))
    return synonyms

def get_antonyms(word):
    """Get antonyms for a word from WordNet."""
    antonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.antonyms():
                for antonym in lemma.antonyms():
                    if antonym.name() not in antonyms:
                        antonyms.append(antonym.name().replace('_', ' '))
    return antonyms

# List of common words to create our thesaurus dataset
common_words = [
    "happy", "sad", "big", "small", "fast", "slow", "beautiful", "ugly",
    "intelligent", "stupid", "good", "bad", "strong", "weak", "rich", "poor",
    "hot", "cold", "old", "new", "easy", "difficult", "loud", "quiet",
    "bright", "dark", "clean", "dirty", "healthy", "sick", "brave", "afraid",
    "kind", "cruel", "polite", "rude", "honest", "dishonest", "active", "lazy"
]

# Collect data
thesaurus_data = []
for word in common_words:
    synonyms = get_synonyms(word)
    antonyms = get_antonyms(word)
    
    if synonyms:  # Only add words that have synonyms
        thesaurus_data.append({
            "word": word,
            "synonyms": synonyms[:10],  # Limit to top 10 synonyms
            "antonyms": antonyms
        })

# Convert to DataFrame for easier manipulation
df = pd.DataFrame(thesaurus_data)
df.head()

## 2. Data Cleaning and Preprocessing

Let's clean and preprocess our data:

In [ ]:
def clean_word_list(word_list):
    """Clean a list of words by removing duplicates and empty strings."""
    return [word.strip().lower() for word in word_list if word.strip()]

# Clean the data
df['synonyms'] = df['synonyms'].apply(clean_word_list)
df['antonyms'] = df['antonyms'].apply(clean_word_list)

# Remove rows with no synonyms
df = df[df['synonyms'].apply(len) > 0].reset_index(drop=True)

# Check the cleaned data
print(f"Dataset size: {len(df)} words")
print(f"Average number of synonyms per word: {df['synonyms'].apply(len).mean():.2f}")
print(f"Average number of antonyms per word: {df['antonyms'].apply(len).mean():.2f}")

# Display a sample
df.sample(5)

## 3. Data Formatting for Different Fine-Tuning Approaches

Different fine-tuning approaches require different data formats. Let's prepare our data for various approaches:

### 3.1 Instruction Tuning Format

For instruction tuning (used in LoRA, QLoRA, etc.), we format the data as instruction-response pairs:

In [ ]:
def format_for_instruction_tuning(df):
    """Format data for instruction tuning."""
    instruction_data = []
    
    # Format synonym examples
    for _, row in df.iterrows():
        instruction_data.append({
            "instruction": f"List synonyms for the word '{row['word']}'.",
            "response": ", ".join(row['synonyms'])
        })
        
        # Add antonym examples if available
        if row['antonyms']:
            instruction_data.append({
                "instruction": f"List antonyms for the word '{row['word']}'.",
                "response": ", ".join(row['antonyms'])
            })
    
    return pd.DataFrame(instruction_data)

# Create instruction tuning dataset
instruction_df = format_for_instruction_tuning(df)
instruction_df.head()

### 3.2 Causal Language Modeling Format

For causal language modeling (traditional fine-tuning), we format the data as continuous text:

In [ ]:
def format_for_causal_lm(df):
    """Format data for causal language modeling."""
    texts = []
    
    for _, row in df.iterrows():
        # Format synonym examples
        texts.append(f"The word '{row['word']}' has the following synonyms: {', '.join(row['synonyms'])}.")
        
        # Add antonym examples if available
        if row['antonyms']:
            texts.append(f"The word '{row['word']}' has the following antonyms: {', '.join(row['antonyms'])}.")
    
    return pd.DataFrame({"text": texts})

# Create causal LM dataset
causal_lm_df = format_for_causal_lm(df)
causal_lm_df.head()

### 3.3 Sequence-to-Sequence Format

For sequence-to-sequence models (T5, BART, etc.), we format the data as input-output pairs:

In [ ]:
def format_for_seq2seq(df):
    """Format data for sequence-to-sequence models."""
    seq2seq_data = []
    
    for _, row in df.iterrows():
        # Format synonym examples
        seq2seq_data.append({
            "input": f"synonyms: {row['word']}",
            "output": ", ".join(row['synonyms'])
        })
        
        # Add antonym examples if available
        if row['antonyms']:
            seq2seq_data.append({
                "input": f"antonyms: {row['word']}",
                "output": ", ".join(row['antonyms'])
            })
    
    return pd.DataFrame(seq2seq_data)

# Create sequence-to-sequence dataset
seq2seq_df = format_for_seq2seq(df)
seq2seq_df.head()

## 4. Train-Test Split

Let's split our data into training and testing sets:

In [ ]:
def split_dataset(df, test_size=0.2, random_state=42):
    """Split a DataFrame into training and testing sets."""
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

# Split the instruction tuning dataset
train_instruction_df, test_instruction_df = split_dataset(instruction_df)

print(f"Training set size: {len(train_instruction_df)}")
print(f"Testing set size: {len(test_instruction_df)}")

## 5. Convert to Hugging Face Dataset Format

Let's convert our data to the Hugging Face Dataset format, which is required for fine-tuning with the Transformers library:

In [ ]:
def convert_to_hf_dataset(train_df, test_df):
    """Convert DataFrames to Hugging Face Dataset format."""
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)
    
    return DatasetDict({
        "train": train_dataset,
        "test": test_dataset
    })

# Convert instruction tuning dataset
instruction_dataset = convert_to_hf_dataset(train_instruction_df, test_instruction_df)
instruction_dataset

## 6. Tokenization

Now, let's tokenize our data for a specific model:

In [ ]:
def tokenize_instruction_dataset(dataset, model_name="gpt2", max_length=128):
    """Tokenize an instruction dataset for a specific model."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Set padding token if not set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    def tokenize_function(examples):
        # Format as instruction-response pairs
        texts = [
            f"### Instruction: {instruction}\n\n### Response: {response}"
            for instruction, response in zip(examples["instruction"], examples["response"])
        ]
        
        # Tokenize the texts
        tokenized = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        # Set the labels to be the same as the inputs for causal LM
        tokenized["labels"] = tokenized["input_ids"].clone()
        
        return tokenized
    
    # Tokenize the dataset
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset["train"].column_names
    )
    
    return tokenized_dataset

# Tokenize the instruction dataset for GPT-2
tokenized_instruction_dataset = tokenize_instruction_dataset(instruction_dataset)
tokenized_instruction_dataset

## 7. Save the Datasets

Let's save our datasets for later use:

In [ ]:
# Save the raw thesaurus data
thesaurus_data_dict = {
    "train": df.iloc[:int(len(df)*0.8)].to_dict(orient="records"),
    "test": df.iloc[int(len(df)*0.8):].to_dict(orient="records")
}

with open('thesaurus_dataset.json', 'w') as f:
    json.dump(thesaurus_data_dict, f, indent=2)

# Save the instruction dataset
instruction_dataset.save_to_disk("instruction_dataset")

# Save the tokenized dataset
tokenized_instruction_dataset.save_to_disk("tokenized_instruction_dataset")

print("Datasets saved successfully!")

## 8. Data Augmentation

Let's augment our data to increase its size and diversity:

In [ ]:
def augment_instruction_data(df):
    """Augment instruction data with variations."""
    augmented_data = []
    
    for _, row in df.iterrows():
        # Original instruction
        augmented_data.append({
            "instruction": row["instruction"],
            "response": row["response"]
        })
        
        # Variation 1: Different phrasing
        if "List synonyms" in row["instruction"]:
            word = row["instruction"].split("'")[1]
            augmented_data.append({
                "instruction": f"What are some synonyms for '{word}'?",
                "response": row["response"]
            })
            augmented_data.append({
                "instruction": f"Give me words that mean the same as '{word}'.",
                "response": row["response"]
            })
        
        # Variation 2: Different phrasing for antonyms
        if "List antonyms" in row["instruction"]:
            word = row["instruction"].split("'")[1]
            augmented_data.append({
                "instruction": f"What are some antonyms for '{word}'?",
                "response": row["response"]
            })
            augmented_data.append({
                "instruction": f"Give me words that mean the opposite of '{word}'.",
                "response": row["response"]
            })
    
    return pd.DataFrame(augmented_data)

# Augment the instruction data
augmented_instruction_df = augment_instruction_data(instruction_df)
print(f"Original dataset size: {len(instruction_df)}")
print(f"Augmented dataset size: {len(augmented_instruction_df)}")

# Display some examples
augmented_instruction_df.sample(5)

## 9. Data Quality Checks

Let's perform some quality checks on our data:

In [ ]:
def check_data_quality(df, column):
    """Check the quality of a text column in a DataFrame."""
    # Check for empty strings
    empty_count = df[column].apply(lambda x: x.strip() == "").sum()
    print(f"Empty {column}: {empty_count}")
    
    # Check for very short texts
    short_count = df[column].apply(lambda x: len(x.split()) < 3).sum()
    print(f"Very short {column} (< 3 words): {short_count}")
    
    # Check for very long texts
    long_count = df[column].apply(lambda x: len(x.split()) > 50).sum()
    print(f"Very long {column} (> 50 words): {long_count}")
    
    # Check for duplicates
    duplicate_count = len(df) - len(df[column].unique())
    print(f"Duplicate {column}: {duplicate_count}")

# Check instruction quality
print("Checking instruction quality:")
check_data_quality(augmented_instruction_df, "instruction")

print("\nChecking response quality:")
check_data_quality(augmented_instruction_df, "response")

## 10. Preparing Data for Different Model Sizes

Different model sizes have different data requirements. Let's prepare subsets of our data for different model sizes:

In [ ]:
def create_size_variants(df, sizes=[100, 500, 1000]):
    """Create smaller variants of the dataset for different model sizes."""
    variants = {}
    
    for size in sizes:
        if size < len(df):
            # Sample without replacement
            variants[f"size_{size}"] = df.sample(size, random_state=42).reset_index(drop=True)
        else:
            print(f"Warning: Requested size {size} is larger than dataset size {len(df)}")
    
    return variants

# Create size variants
size_variants = create_size_variants(augmented_instruction_df)

# Display the sizes
for name, variant in size_variants.items():
    print(f"{name}: {len(variant)} examples")

## 11. Save the Final Datasets

Let's save our final datasets for use in fine-tuning:

In [ ]:
# Save the augmented instruction data
augmented_instruction_df.to_csv("augmented_instruction_data.csv", index=False)

# Save the size variants
for name, variant in size_variants.items():
    variant.to_csv(f"{name}_instruction_data.csv", index=False)

# Save as JSON for easy loading in fine-tuning scripts
train_df, test_df = split_dataset(augmented_instruction_df)
final_dataset = {
    "train": train_df.to_dict(orient="records"),
    "test": test_df.to_dict(orient="records")
}

with open('final_thesaurus_dataset.json', 'w') as f:
    json.dump(final_dataset, f, indent=2)

print("Final datasets saved successfully!")

## Conclusion

In this tutorial, you've learned how to:

1. Collect thesaurus data from WordNet
2. Clean and preprocess the data
3. Format the data for different fine-tuning approaches
4. Split the data into training and testing sets
5. Convert the data to Hugging Face Dataset format
6. Tokenize the data for a specific model
7. Augment the data to increase its size and diversity
8. Perform quality checks on the data
9. Prepare subsets of the data for different model sizes
10. Save the final datasets for use in fine-tuning

Proper data preparation is crucial for successful fine-tuning. By following these steps, you can ensure that your data is high-quality, well-formatted, and appropriate for your specific fine-tuning approach and model size.

## References

- [Hugging Face Datasets Documentation](https://huggingface.co/docs/datasets/index)
- [NLTK WordNet Documentation](https://www.nltk.org/howto/wordnet.html)
- [Data Preparation for NLP](https://huggingface.co/blog/nlp-data-preparation)
- [Best Practices for Fine-Tuning Language Models](https://huggingface.co/blog/how-to-train)